# Docker Deployment for LLM Inference

**Stage 3 - Production | Notebook 2**

## Learning Objectives
- Master Docker for ML workloads (GPU, model volumes, multi-stage builds)
- Deploy vLLM inference service with Docker Compose
- Configure GPU passthrough, resource constraints, and health checks
- Optimize images for size and build speed
- Push images to container registries

In [ ]:
!pip install docker compose

## 1. Docker Basics Refresher for ML

Key instructions and why they matter for ML workloads:

| Instruction | Purpose | ML-Specific Note |
|-------------|---------|------------------|
| `FROM` | Base image | Use `nvidia/cuda:12.4.0-devel-ubuntu22.04` for GPU builds |
| `RUN` | Execute build-time commands | Install Python deps, compile custom kernels |
| `COPY` | Copy files into image | Copy model configs, source code |
| `CMD` | Default runtime command | Launch the inference server |
| `ENV` | Set environment variables | `CUDA_VISIBLE_DEVICES`, `HF_HOME` |
| `VOLUME` | Declare mount point | Model cache directories |
| `WORKDIR` | Set working directory | Standardize paths for scripts |

In [ ]:
# Inspect NVIDIA CUDA image layers
!docker pull nvidia/cuda:12.4.0-runtime-ubuntu22.04
!docker history nvidia/cuda:12.4.0-runtime-ubuntu22.04 | head -20

## 2. Multi-Stage Dockerfile for vLLM Inference

Multi-stage builds separate the build environment from the runtime, reducing final image size.

In [ ]:
%%writefile Dockerfile.vllm
# ============================================================
# Stage 1: Build dependencies
# ============================================================
FROM nvidia/cuda:12.4.0-devel-ubuntu22.04 AS builder

ENV PYTHONUNBUFFERED=1 \
    DEBIAN_FRONTEND=noninteractive \
    PIP_NO_CACHE_DIR=1

RUN apt-get update && apt-get install -y --no-install-recommends \
    python3.10 python3.10-dev python3-pip git && \
    rm -rf /var/lib/apt/lists/*

# Install vLLM from source for better compatibility
RUN pip install --upgrade pip setuptools wheel
RUN pip install vllm==0.6.3.post1

# ============================================================
# Stage 2: Runtime image
# ============================================================
FROM nvidia/cuda:12.4.0-runtime-ubuntu22.04 AS runtime

ENV PYTHONUNBUFFERED=1 \
    DEBIAN_FRONTEND=noninteractive \
    HF_HOME=/models/.cache \
    NVIDIA_VISIBLE_DEVICES=all

RUN apt-get update && apt-get install -y --no-install-recommends \
    python3.10 python3-pip curl && \
    rm -rf /var/lib/apt/lists/*

# Copy installed packages from builder
COPY --from=builder /usr/local/lib/python3.10/dist-packages /usr/local/lib/python3.10/dist-packages
COPY --from=builder /usr/local/bin /usr/local/bin

RUN useradd -m -u 1000 vllm
USER vllm
WORKDIR /app

# Default vLLM serve command (overridable)
CMD ["python3", "-m", "vllm.entrypoints.openai.api_server", \
     "--host", "0.0.0.0", "--port", "8000"]

EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=10s --start-period=60s --retries=3 \
  CMD curl -f http://localhost:8000/health || exit 1

## 3. Docker Compose: vLLM + API Gateway

Orchestrate multiple containers: vLLM server, Nginx reverse proxy, and optional model downloader init container.

In [ ]:
%%writefile docker-compose.yml
services:
  # --- vLLM Inference Server ---
  vllm:
    image: vllm/vllm-openai:latest
    runtime: nvidia
    environment:
      - CUDA_VISIBLE_DEVICES=0,1
      - HF_HOME=/models/.cache
    volumes:
      - /mnt/models:/models:ro          # Read-only model mount
      - vllm_cache:/models/.cache       # Writable cache for HF downloads
    ports:
      - "8000:8000"
    command:
      - "--model"
      - "/models/Meta-Llama-3-8B-Instruct"
      - "--max-model-len"
      - "8192"
      - "--gpu-memory-utilization"
      - "0.90"
    deploy:
      resources:
        reservations:
          devices:
            - driver: nvidia
              count: 2
              capabilities: [gpu]
    shm_size: "8gb"
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
      start_period: 120s

  # --- API Gateway (Nginx) ---
  nginx:
    image: nginx:alpine
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf:ro
    ports:
      - "80:80"
    depends_on:
      vllm:
        condition: service_healthy

  # --- Prometheus Metrics Sidecar ---
  prometheus:
    image: prom/prometheus:latest
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml:ro
    ports:
      - "9090:9090"

volumes:
  vllm_cache:

## 4. GPU Passthrough with nvidia-docker

Three must-have components for GPU containers:
1. **NVIDIA Container Toolkit** (`nvidia-container-toolkit`)
2. **`--gpus` flag** or `runtime: nvidia` in compose
3. **NVIDIA drivers** on the host (check with `nvidia-smi`)

In [ ]:
# Check host GPU availability
!nvidia-smi

# Verify nvidia-docker runtime is registered
!docker info | grep -A5 "Runtimes"

In [ ]:
# Install NVIDIA Container Toolkit (on Ubuntu host)
# distribution=$(. /etc/os-release;echo $ID$VERSION_ID)
# curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg
# curl -s -L https://nvidia.github.io/libnvidia-container/$distribution/libnvidia-container.list | \
#   sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
#   sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list
# sudo apt-get update && sudo apt-get install -y nvidia-container-toolkit
# sudo systemctl restart docker

# RUNNING WITH GPU:
# docker run --gpus all vllm/vllm-openai:latest
# docker run --gpus '"device=0,1"' vllm/vllm-openai:latest  # specific GPUs

## 5. Model Volume Mounting Patterns

Common patterns for mounting model weights:

In [ ]:
# Pattern 1: Read-only host mount (production)
# docker run -v /data/models:/models:ro vllm-server --model /models/llama-8b

# Pattern 2: Docker volume for HF cache (shared across restarts)
# docker volume create hf_cache
# docker run -v hf_cache:/root/.cache/huggingface vllm-server --model meta-llama/Llama-3-8B

# Pattern 3: Init container to pre-download models
init_container_script = """
FROM python:3.10-slim AS model-downloader
RUN pip install huggingface_hub
ARG MODEL_ID=meta-llama/Meta-Llama-3-8B-Instruct
RUN python -c "from huggingface_hub import snapshot_download; snapshot_download('$MODEL_ID')"

# Then mount the same volume into vLLM container
"""
print(init_container_script)

## 6. Resource Constraints

Critical for GPU workloads where shared memory and memory limits matter.

In [ ]:
# Resource constraint reference for Docker run:
resource_cheatsheet = """
# GPU allocation
--gpus '"device=0,1"'          # Use specific GPUs
--gpus all                      # All GPUs

# Memory limits
--memory="16g"                  # Hard memory limit
--memory-swap="16g"             # Disable swap (same as memory)
--shm-size="8g"                 # Shared memory for NCCL/torch

# CPU limits
--cpus="8.0"                   # CPU quota
--cpuset-cpus="0-7"             # Pin to specific cores

# Disk I/O
--device-read-bps=/dev/nvme0n1:500mb   # Limit read bandwidth
--device-write-bps=/dev/nvme0n1:500mb  # Limit write bandwidth

# Full example
docker run --gpus all \\
  --memory="32g" \\
  --shm-size="8g" \\
  --cpus="8.0" \\
  -v /models:/models:ro \\
  vllm-server
"""
print(resource_cheatsheet)

## 7. Image Optimization

### Layer Ordering Best Practices
- Put least-frequently-changing instructions first (OS packages)
- Copy only what's needed (`.dockerignore`)
- Combine RUN commands to reduce layers
- Remove caches in the same layer (`rm -rf /var/lib/apt/lists/*`)

In [ ]:
%%writefile .dockerignore
# Python
__pycache__/
*.py[cod]
*.egg-info/
.eggs/
dist/
build/
*.whl

# Virtual environments
.venv/
venv/
env/

# IDE and editor files
.vscode/
.idea/
*.swp
*.swo

# Git
.git/
.gitignore

# Tests and documentation
tests/
docs/
*.md

# Data and models (mounted at runtime)
models/
data/
checkpoints/
*.bin
*.safetensors

# Docker
Dockerfile*
docker-compose*.yml
.dockerignore

# Misc
*.log
.env*

In [ ]:
# Check image size before and after optimization
!docker images --format "table {{.Repository}}\t{{.Tag}}\t{{.Size}}" | head -10

# Analyze layers with dive (install: https://github.com/wagoodman/dive)
# !dive vllm/vllm-openai:latest

## 8. Pushing to Docker Hub / Private Registry

In [ ]:
# Docker Hub
# docker tag vllm-server:latest yourusername/vllm-server:v1.0.0
# docker push yourusername/vllm-server:v1.0.0

# AWS ECR
# aws ecr get-login-password --region us-east-1 | docker login --username AWS --password-stdin 123456789.dkr.ecr.us-east-1.amazonaws.com
# docker tag vllm-server:latest 123456789.dkr.ecr.us-east-1.amazonaws.com/vllm-server:v1.0.0
# docker push 123456789.dkr.ecr.us-east-1.amazonaws.com/vllm-server:v1.0.0

# GCP Artifact Registry
# gcloud auth configure-docker us-central1-docker.pkg.dev
# docker tag vllm-server:latest us-central1-docker.pkg.dev/my-project/vllm-repo/vllm-server:v1.0.0
# docker push us-central1-docker.pkg.dev/my-project/vllm-repo/vllm-server:v1.0.0

print("Uncomment the relevant block for your registry.")

## 9. ENV Configuration for dev/staging/prod

Separate configuration from code using environment variables.

In [ ]:
%%writefile .env.dev
ENVIRONMENT=dev
MODEL_PATH=/models/Llama-3-8B
GPU_MEMORY_UTILIZATION=0.85
MAX_MODEL_LEN=4096
LOG_LEVEL=DEBUG
API_KEY=dev-key-not-secret
OTEL_ENABLED=false

%%writefile .env.prod
ENVIRONMENT=prod
MODEL_PATH=/models/Llama-3-70B
GPU_MEMORY_UTILIZATION=0.92
MAX_MODEL_LEN=32768
LOG_LEVEL=INFO
OTEL_ENABLED=true
OTEL_EXPORTER=otlp://jaeger:4317
# API_KEY set via Docker secret / K8s Secret, not here

In [ ]:
# Compose file that loads env files:
compose_with_env = """
services:
  vllm:
    image: vllm-server:latest
    env_file:
      - .env.${ENVIRONMENT:-dev}   # ENVIRONMENT=prod docker compose up
    environment:
      - API_KEY=${API_KEY:?API_KEY must be set}  # Must be provided
"""
print(compose_with_env)

## 10. HEALTHCHECK in Docker

vLLM exposes `/health` endpoint. Configure Docker to use it.

In [ ]:
# HEALTHCHECK in Dockerfile:
healthcheck_dockerfile = """
# Basic check — endpoint returns 200 when server is ready
HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \
  CMD curl -f http://localhost:8000/health || exit 1

# Advanced check — also verify /v1/models returns a model
HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=3 \
  CMD curl -sf http://localhost:8000/v1/models | python3 -c "import sys,json; d=json.load(sys.stdin); assert len(d['data'])>0" || exit 1
"""
print(healthcheck_dockerfile)

# Check health status of running containers
# docker ps --format "table {{.Names}}\t{{.Status}}"

## 11. Docker Debugging

Essential commands for troubleshooting containerized ML services.

In [ ]:
# ── Debugging Cheat Sheet ──
print("""
# View logs (follow)
docker logs -f <container_name>

# View last 100 lines with timestamps
docker logs --tail 100 -t <container_name>

# Execute shell inside running container
docker exec -it <container_name> /bin/bash

# Run nvidia-smi inside container
docker exec <container_name> nvidia-smi

# Inspect container metadata (mounts, env, network)
docker inspect <container_name> | jq '.[0].Mounts'
docker inspect <container_name> | jq '.[0].Config.Env'

# Check resource usage
docker stats <container_name>

# Check GPU utilization inside container
docker exec <container_name> nvidia-smi dmon -s pucv -d 2

# Copy files to/from container
docker cp <container_name>:/app/logs ./local-logs

# View image layers
docker history <image_name>

# Prune unused images/containers/volumes
docker system prune -a --volumes
""")

## 12. Exercise: Write Dockerfile for Training Environment

**TODO:** Create a multi-stage Dockerfile for a distributed training environment that:
- Uses `nvidia/cuda:12.4.0-devel-ubuntu22.04` as base
- Installs PyTorch with CUDA 12.4, DeepSpeed, and Flash Attention 2
- Configures NCCL for multi-node communication
- Sets up SSH for distributed launch
- Adds a non-root user
- Uses HEALTHCHECK to verify GPU accessibility
- Keeps the final image under 12GB

In [ ]:
# TODO: Write your Dockerfile below
%%writefile Dockerfile.training
# ============================================================
# Stage 1: Builder
# ============================================================
FROM nvidia/cuda:12.4.0-devel-ubuntu22.04 AS builder

# YOUR CODE HERE
# - Install Python 3.10, git, build tools
# - Install PyTorch 2.4 with CUDA 12.4
# - Install DeepSpeed
# - Install Flash Attention 2
# - Install NCCL-related tools

# ============================================================
# Stage 2: Runtime
# ============================================================
FROM nvidia/cuda:12.4.0-runtime-ubuntu22.04 AS runtime

# YOUR CODE HERE
# - Copy built packages from Stage 1
# - Set up non-root user
# - Configure SSH for multi-node
# - Set NCCL environment variables
# - Add HEALTHCHECK with nvidia-smi
# - Set default entrypoint

print("Replace the placeholder comments with actual Dockerfile instructions.")

## Summary

In this notebook, you learned:
- **Multi-stage Dockerfiles** for efficient ML images
- **Docker Compose** for orchestrating vLLM + API gateway
- **GPU passthrough** with nvidia-container-toolkit
- **Model volume patterns** (read-only mounts, init containers)
- **Resource constraints** (`--gpus`, `--memory`, `--shm-size`)
- **Image optimization** (layer ordering, `.dockerignore`)
- **Container registries** (Docker Hub, ECR, GAR)
- **ENV-based config** for environment-specific settings
- **HEALTHCHECK** patterns for vLLM
- **Debugging** commands for containerized ML

**Next:** Kubernetes orchestration for production deployment.